سلول ۱ — Import و مسیرها

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

from xgboost import XGBClassifier
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".")

DATA_ML_DIR = PROJECT_ROOT / "Data_ml"
PAIR_FEATURE_DIR = DATA_ML_DIR / "pair_features"
RESULT_DIR = DATA_ML_DIR / "cross_model_fusion_results"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

سلول ۲ — تعریف مدل‌ها و Fusionها

In [ ]:
SELECTED_MODELS = {
    "esm650": "esm2_t33_650M_UR50D",
    "esm3b": "esm2_t36_3B_UR50D",
    "protbert_bfd": "prot_bert_bfd",
}

CROSS_MODEL_FUSIONS = {
    "esm650__protbert_bfd": [
        "esm2_t33_650M_UR50D",
        "prot_bert_bfd",
    ],
    "esm3b__protbert_bfd": [
        "esm2_t36_3B_UR50D",
        "prot_bert_bfd",
    ],
    "esm650__esm3b": [
        "esm2_t33_650M_UR50D",
        "esm2_t36_3B_UR50D",
    ],
    "esm650__esm3b__protbert_bfd": [
        "esm2_t33_650M_UR50D",
        "esm2_t36_3B_UR50D",
        "prot_bert_bfd",
    ],
}

FEATURE_TYPE_TO_USE = "fusion"  # از fusion داخلی هر مدل استفاده می‌کنیم

سلول ۳ — Loader و QC ترتیب ردیف‌ها

In [ ]:
def load_model_feature(model_name, feature_type="fusion"):
    model_dir = PAIR_FEATURE_DIR / model_name
    
    meta = pd.read_csv(
        model_dir / "meta.csv",
        dtype=str,
        low_memory=False,
    )
    
    X = pd.read_parquet(
        model_dir / f"{feature_type}.features.parquet"
    )
    
    return X, meta


def build_cross_model_feature(fusion_name, model_names, feature_type="fusion"):
    X_list = []
    meta_list = []
    
    for model_name in model_names:
        X, meta = load_model_feature(model_name, feature_type)
        X_list.append(X)
        meta_list.append(meta)
    
    ref_meta = meta_list[0]
    
    for model_name, meta in zip(model_names[1:], meta_list[1:]):
        assert meta["pair_id"].tolist() == ref_meta["pair_id"].tolist(), f"pair_id order mismatch: {model_name}"
        assert meta["label"].astype(int).tolist() == ref_meta["label"].astype(int).tolist(), f"label mismatch: {model_name}"
        assert meta["group_id"].tolist() == ref_meta["group_id"].tolist(), f"group_id mismatch: {model_name}"
    
    X_cross = pd.concat(X_list, axis=1)
    
    # ستون‌ها از قبل prefix دارند، ولی باز هم چک می‌کنیم.
    assert X_cross.columns.nunique() == X_cross.shape[1], "Duplicate feature columns detected."
    
    y = ref_meta["label"].astype(int).values
    groups = ref_meta["group_id"].values
    
    return X_cross, y, groups, ref_meta

سلول ۴ — ساخت و ذخیره چهار Fusion

In [ ]:
fusion_qc_rows = []

for fusion_name, model_names in CROSS_MODEL_FUSIONS.items():
    print("=" * 100)
    print("Building:", fusion_name)
    print(model_names)
    
    X_cross, y, groups, meta = build_cross_model_feature(
        fusion_name=fusion_name,
        model_names=model_names,
        feature_type=FEATURE_TYPE_TO_USE,
    )
    
    out_dir = RESULT_DIR / fusion_name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    X_cross.to_parquet(
        out_dir / "cross_model_fusion.features.parquet",
        index=False,
    )
    
    meta.to_csv(
        out_dir / "meta.csv",
        index=False,
    )
    
    qc = {
        "fusion_name": fusion_name,
        "models": ";".join(model_names),
        "n_pairs": X_cross.shape[0],
        "n_features": X_cross.shape[1],
        "n_positive": int((y == 1).sum()),
        "n_negative": int((y == 0).sum()),
        "nan_count": int(X_cross.isna().sum().sum()),
        "inf_count": int(np.isinf(X_cross.to_numpy(dtype=np.float32)).sum()),
        "duplicate_feature_cols": int(X_cross.shape[1] - X_cross.columns.nunique()),
        "n_unique_pair_id": meta["pair_id"].nunique(),
        "n_duplicate_pair_id": int(meta.duplicated("pair_id").sum()),
    }
    
    fusion_qc_rows.append(qc)
    
fusion_feature_qc = pd.DataFrame(fusion_qc_rows)
display(fusion_feature_qc)

fusion_feature_qc.to_csv(
    QC_DIR / "day10_cross_model_fusion_feature_qc.csv",
    index=False,
)

سلول ۵ — مدل‌های Baseline

In [ ]:
def build_logistic():
    return LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42,
    )


def build_rf():
    return RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )


def build_xgb():
    return XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

سلول ۶ — Evaluation با GroupKFold

In [ ]:
def evaluate_model(estimator, X, y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    
    fold_rows = []
    
    for fold_idx, (tr, te) in enumerate(gkf.split(X, y, groups)):
        Xtr = X.iloc[tr]
        Xte = X.iloc[te]
        
        ytr = y[tr]
        yte = y[te]
        
        estimator.fit(Xtr, ytr)
        
        prob = estimator.predict_proba(Xte)[:, 1]
        pred = (prob >= 0.5).astype(int)
        
        fold_rows.append({
            "fold": fold_idx,
            "roc_auc": roc_auc_score(yte, prob),
            "pr_auc": average_precision_score(yte, prob),
            "f1": f1_score(yte, pred, zero_division=0),
            "accuracy": accuracy_score(yte, pred),
            "precision": precision_score(yte, pred, zero_division=0),
            "recall": recall_score(yte, pred, zero_division=0),
            "brier": brier_score_loss(yte, prob),
            "n_test": len(te),
            "n_test_pos": int((yte == 1).sum()),
            "n_test_neg": int((yte == 0).sum()),
        })
    
    return pd.DataFrame(fold_rows)

سلول ۷ — اجرای ۱۲ Benchmark

In [ ]:
all_results = []
all_fold_results = []

for fusion_name, model_names in CROSS_MODEL_FUSIONS.items():
    print()
    print("=" * 100)
    print("Fusion:", fusion_name)
    
    fusion_dir = RESULT_DIR / fusion_name
    
    X = pd.read_parquet(
        fusion_dir / "cross_model_fusion.features.parquet"
    )
    
    meta = pd.read_csv(
        fusion_dir / "meta.csv",
        dtype=str,
        low_memory=False,
    )
    
    y = meta["label"].astype(int).values
    groups = meta["group_id"].values
    
    estimators = {
        "logistic": build_logistic(),
        "random_forest": build_rf(),
        "xgboost": build_xgb(),
    }
    
    for algo_name, est in estimators.items():
        print("running:", fusion_name, algo_name)
        
        fold_df = evaluate_model(
            estimator=est,
            X=X,
            y=y,
            groups=groups,
            n_splits=5,
        )
        
        fold_df["fusion_name"] = fusion_name
        fold_df["models"] = ";".join(model_names)
        fold_df["algorithm"] = algo_name
        
        all_fold_results.append(fold_df)
        
        result_row = {
            "fusion_name": fusion_name,
            "models": ";".join(model_names),
            "algorithm": algo_name,
            "n_pairs": len(meta),
            "n_features": X.shape[1],
        }
        
        for metric in [
            "roc_auc",
            "pr_auc",
            "f1",
            "accuracy",
            "precision",
            "recall",
            "brier",
        ]:
            result_row[f"{metric}_mean"] = fold_df[metric].mean()
            result_row[f"{metric}_std"] = fold_df[metric].std()
        
        all_results.append(result_row)

cross_fusion_results = pd.DataFrame(all_results)
cross_fusion_fold_results = pd.concat(all_fold_results, ignore_index=True)

display(cross_fusion_results)

cross_fusion_results.to_csv(
    RESULT_DIR / "cross_model_fusion_results_all.csv",
    index=False,
)

cross_fusion_fold_results.to_csv(
    RESULT_DIR / "cross_model_fusion_fold_results_all.csv",
    index=False,
)

سلول ۸ — Leaderboard

In [ ]:
cross_fusion_leaderboard = (
    cross_fusion_results
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)

display(cross_fusion_leaderboard)

cross_fusion_leaderboard.to_csv(
    RESULT_DIR / "cross_model_fusion_leaderboard_pr_auc.csv",
    index=False,
)

سلول ۹ — مقایسه با بهترین مدل روز ۹

In [ ]:
DAY9_LEADERBOARD_PATH = DATA_ML_DIR / "baseline_results" / "leaderboard_pr_auc.csv"

day9_leaderboard = pd.read_csv(DAY9_LEADERBOARD_PATH)

best_day9 = day9_leaderboard.iloc[0].copy()
best_day10 = cross_fusion_leaderboard.iloc[0].copy()

comparison = pd.DataFrame([
    {
        "stage": "best_day9_single_model",
        "name": f"{best_day9['embedding_model']} | {best_day9['feature_type']} | {best_day9['algorithm']}",
        "roc_auc_mean": best_day9["roc_auc_mean"],
        "pr_auc_mean": best_day9["pr_auc_mean"],
        "f1_mean": best_day9["f1_mean"],
        "brier_mean": best_day9["brier_mean"],
    },
    {
        "stage": "best_day10_cross_model_fusion",
        "name": f"{best_day10['fusion_name']} | {best_day10['algorithm']}",
        "roc_auc_mean": best_day10["roc_auc_mean"],
        "pr_auc_mean": best_day10["pr_auc_mean"],
        "f1_mean": best_day10["f1_mean"],
        "brier_mean": best_day10["brier_mean"],
    },
])

comparison["delta_pr_auc_vs_day9"] = comparison["pr_auc_mean"] - comparison.loc[0, "pr_auc_mean"]
comparison["delta_roc_auc_vs_day9"] = comparison["roc_auc_mean"] - comparison.loc[0, "roc_auc_mean"]

display(comparison)

comparison.to_csv(
    RESULT_DIR / "day9_vs_day10_best_model_comparison.csv",
    index=False,
)

In [ ]:
from pathlib import Path
import os

for root, dirs, files in os.walk(PROJECT_ROOT):
    for f in files:
        if "fusion" in f.lower():
            print(Path(root) / f)